# **DEEP NEURAL NETWORK**
We will rewrite the three-layer neural network to be easily expandable to any number of layers and be able to handle more advanced activation functions, initial values, and optimization methods. By doing this kind of scratching, we hope to be able to imagine the internal workings of various frameworks as we use them in the future.

# Problem 1: Classification of fully connected layers
We classify the fully connected layer.
The weights and biases are initialized in the constructor, and then forward and backward methods are prepared. By storing the weights W, bias B, and forward input X as instance variables, complicated input and output is no longer necessary.

If we pass an instance initializer of the initialization method to the constructor, the initialization will be performed by that. If we change the instance you pass, we can change the initialization method.

We also pass the self instance as an argument self.optimizer.update(self)You can use this to update the weights of the layer. There are multiple values required for the update, but they can all be instance variables of the fully connected layer.

Classes of initialization methods and optimization methods will be described later.

In [1]:
class FC:
    def __init__(self, n_nodes1, n_nodes2, initializer, optimizer=None):
        self.n_nodes1 = n_nodes1
        self.n_nodes2 = n_nodes2
        self.initializer = initializer
        self.optimizer = optimizer

        # Initialize weights and biases
        self.W = self.initializer.initialize((n_nodes1, n_nodes2))
        self.B = self.initializer.initialize((1, n_nodes2))

        # Cache for forward/backward
        self.X = None
        self.dW = None
        self.dB = None

    def forward(self, X):
        self.X = X
        out = np.dot(X, self.W) + self.B
        return out

    def backward(self, d_out):
        batch_size = self.X.shape[0]
        self.dW = np.dot(self.X.T, d_out) / batch_size
        self.dB = np.sum(d_out, axis=0, keepdims=True) / batch_size
        d_input = np.dot(d_out, self.W.T)

        if self.optimizer:
            self.optimizer.update(self)

        return d_input

# Problem 2: Classifying the initialization method
Put the initialization code into a class.

As mentioned above, we can pass an instance of the initialization method to the constructor of the fully connected layer. By receiving the standard deviation value (sigma) in the constructor, we will not need to pass this value (sigma) into the fully connected layer class.

We will name the initialization method we have dealt with so far as the SimpleInitializer class.

In [2]:
import numpy as np

class SimpleInitializer:

    def __init__(self, sigma):
        self.sigma = sigma

    def W(self, n_nodes1, n_nodes2):

        W = self.sigma * np.random.randn(n_nodes1, n_nodes2)
        return W

    def B(self, n_nodes2):

        B = self.sigma * np.random.randn(1, n_nodes2)
        return B

In [3]:
# Update FC Class
class FC:
    def __init__(self, n_nodes1, n_nodes2, initializer, optimizer=None):
        self.n_nodes1 = n_nodes1
        self.n_nodes2 = n_nodes2
        self.initializer = initializer
        self.optimizer = optimizer

        # Use the explicit interface
        self.W = self.initializer.W(n_nodes1, n_nodes2)
        self.B = self.initializer.B(n_nodes2)

        # Cache for forward/backward
        self.X = None
        self.dW = None
        self.dB = None

    def forward(self, X):
        self.X = X
        return np.dot(X, self.W) + self.B

    def backward(self, d_out):
        batch_size = self.X.shape[0]
        self.dW = np.dot(self.X.T, d_out) / batch_size
        self.dB = np.sum(d_out, axis=0, keepdims=True) / batch_size
        d_input = np.dot(d_out, self.W.T)

        if self.optimizer:
            self.optimizer.update(self)

        return d_input

# Problem 3: Classification of optimization methods
Classify the optimization methods.

The optimization method is also passed as an instance to the fully connected layer in the same way as the initialization method. self.optimizer.update(self)It can be updated like in the backwards case. The optimization methods we have dealt with so far are created as SGD class (Stochastic Gradient Descent).

In [4]:
class SGD:

    def __init__(self, lr):
        self.lr = lr

    def update(self, layer):

        layer.W -= self.lr * layer.dW
        layer.B -= self.lr * layer.dB


In [5]:
import numpy as np

class SimpleInitializer:
    
    def __init__(self, sigma):
        self.sigma = sigma

    def W(self, n_nodes1, n_nodes2):
        return self.sigma * np.random.randn(n_nodes1, n_nodes2)

    def B(self, n_nodes2):
        return self.sigma * np.random.randn(1, n_nodes2)


In [6]:
class FC:
    def __init__(self, n_nodes1, n_nodes2, initializer, optimizer=None):
        self.n_nodes1 = n_nodes1
        self.n_nodes2 = n_nodes2
        self.initializer = initializer
        self.optimizer = optimizer.__class__(**optimizer.__dict__) 

        self.W = self.initializer.W(n_nodes1, n_nodes2)
        self.B = self.initializer.B(n_nodes2)

        self.X = None
        self.dW = None
        self.dB = None

    def forward(self, X):
        self.X = X
        return np.dot(X, self.W) + self.B

    def backward(self, d_out):
        batch_size = self.X.shape[0]
        self.dW = np.dot(self.X.T, d_out) / batch_size
        self.dB = np.sum(d_out, axis=0, keepdims=True) / batch_size
        d_input = np.dot(d_out, self.W.T)

        if self.optimizer:
            self.optimizer.update(self)

        return d_input

# Problem 4 Classification of activation functions
Classify the activation functions.

The backpropagation of the softmax function can be simplified by implementing it in a way that also includes the calculation of the cross-entropy error.

We design activation functions with forward(X) and backward(d_out).

In [7]:
class SoftmaxWithLoss:
    def __init__(self):
        self.y = None   # softmax output
        self.t = None   # ground truth (one-hot or index)
        self.loss = None

    def forward(self, X, t):
        self.t = t
        self.y = self._softmax(X)
        self.loss = self._cross_entropy(self.y, self.t)
        return self.loss

    def backward(self, d_out=1):
        batch_size = self.t.shape[0]

        if self.t.size == self.y.size:
            return (self.y - self.t) / batch_size
        # If t is class index
        else:
            dx = self.y.copy()
            dx[np.arange(batch_size), self.t] -= 1
            return dx / batch_size

    def _softmax(self, X):
        X = X - np.max(X, axis=1, keepdims=True)
        exp_X = np.exp(X)
        return exp_X / np.sum(exp_X, axis=1, keepdims=True)

    def _cross_entropy(self, y, t):
        eps = 1e-7
        if y.shape == t.shape:  # one-hot case
            return -np.sum(t * np.log(y + eps)) / y.shape[0]
        else:  # class index case
            return -np.sum(np.log(y[np.arange(y.shape[0]), t] + eps)) / y.shape[0]


In [8]:
class Sigmoid:
    def forward(self, x):
        """Sigmoid activation function"""
        self.output = 1 / (1 + np.exp(-x))
        return self.output
    
    def backward(self, d_out):
        """Derivative of Sigmoid for backpropagation"""
        return d_out * self.output * (1 - self.output)

# Advanced elements
We will implement activation functions, initial weight values, and optimization methods other than those we have seen so far.

# Problem 5: Creating a ReLU class
Implement the ReLU (Rectified Linear Unit), a commonly used activation function today, as a ReLU class.

In [9]:
class ReLU:
    def __init__(self):
        self.mask = None

    def forward(self, X):
        self.mask = (X <= 0)
        out = X.copy()
        out[self.mask] = 0
        return out

    def backward(self, d_out):
        d_out[self.mask] = 0
        return d_out

# Problem 6: Initial weight values
So far, the initial values ​​of the weights and biases have been simply Gaussian distributions, with the standard deviation treated as a hyperparameter. However, it is known what values ​​are best. For sigmoid functions and hyperbolic tangent functions, the initial values ​​of Xavier (or Glorot) are used, and for ReLU, the initial values ​​of He are used.

Create XavierInitializer class and HeInitializer class.

In [10]:
import numpy as np

class XavierInitializer:

    def __init__(self):
        pass

    def W(self, n_nodes1, n_nodes2):
        std = np.sqrt(1.0 / n_nodes1)
        return std * np.random.randn(n_nodes1, n_nodes2)

    def B(self, n_nodes2):
        return np.zeros((1, n_nodes2))  # often initialized as zeros

In [11]:
class HeInitializer:

    def __init__(self):
        pass

    def W(self, n_nodes1, n_nodes2):
        std = np.sqrt(2.0 / n_nodes1)
        return std * np.random.randn(n_nodes1, n_nodes2)

    def B(self, n_nodes2):
        return np.zeros((1, n_nodes2))  # often initialized as zeros

# Problem 7: Optimization methods
It is common to vary the learning rate during the learning process. We create a basic AdaGrad class.

First, let's check the SGD we've been using so far.

$$ W
'
i
=
W_i −α\frac{∂L}{∂W_i}
$$

$$ B
'
i
=
B
i
−
α\frac{∂L}{∂B_i}
$$
$\alpha$ : Learning rate (can be changed for each layer, but basically it is the same for all layers)

$\frac{\partial L}{\partial W_i}$ : Gradient of loss $L$ with respect to $W_i$

$\frac{\partial L}{\partial B_i}$ : Gradient of loss $L$ with respect to $B_i$

Next, we have AdaGrad. I'll skip the formula for the biases, but it does the same thing as the weights.

Gradually decrease the learning rate for that weight by the updated amount. Save the sum of squared gradients $H$ for each iteration and decrease the learning rate by that amount.

The learning rate will be different for each weight.

$$ H'i = H_i + \frac{∂L}{∂W_i} ⊙ \frac{∂L}{∂W_i}$$

$$ W'i = W_i − α(\frac{1}{√(H_i)} ⊙ \frac{∂L}{∂W_i})$$
$H_i$ : For the i-th layer, the sum of squares of the gradients up to the previous iteration (initial value is 0)

$H_i^{\prime}$ : updated $H_i$

In [12]:
import numpy as np

class AdaGrad:
    """
    AdaGrad Optimizer
    Adapts the learning rate for each parameter.
    Parameters
    ----------
    lr : float
        Learning rate (initial)
    """
    def __init__(self, lr=0.01):
        self.lr = lr
        self.h_W = None
        self.h_B = None
        self.epsilon = 1e-7  # To prevent division by zero

    def update(self, layer):
        if self.h_W is None:
            self.h_W = np.zeros_like(layer.W)
            self.h_B = np.zeros_like(layer.B)

        # Accumulate the squared gradients
        self.h_W += layer.dW ** 2
        self.h_B += layer.dB ** 2

        # Update weights and biases
        layer.W -= self.lr * layer.dW / (np.sqrt(self.h_W) + self.epsilon)
        layer.B -= self.lr * layer.dB / (np.sqrt(self.h_B) + self.epsilon)

In [13]:
# Updated FC
class FC:
    def __init__(self, n_input, n_output, initializer, optimizer):
        self.W = initializer.W(n_input, n_output)
        self.B = initializer.B(n_output)
        self.optimizer = optimizer.__class__(lr=optimizer.lr)
        self.dW = None
        self.dB = None
        self.X = None  # input cache
        self.Z = None  # output cache

    def forward(self, X):
        self.X = X
        self.Z = np.dot(X, self.W) + self.B
        return self.Z

    def backward(self, dZ):
        m = self.X.shape[0]
        self.dW = np.dot(self.X.T, dZ) / m
        self.dB = np.sum(dZ, axis=0, keepdims=True) / m
        return np.dot(dZ, self.W.T)

    def update(self):
        self.optimizer.update(self)

# Problem 8: Completing the class
Complete the ScratchDeepNeuralNetrowkClassifier class, which can perform training and estimation in any configuration.

In [14]:
import numpy as np
import pandas as pd

class ScratchDeepNeuralNetworkClassifier:
    def __init__(self, config, initializer, optimizer, loss_fn):
        self.layers = []
        self.loss_fn = loss_fn
        self.config = config

        for in_size, out_size, activation in config:
            self.layers.append(FC(in_size, out_size, initializer, optimizer))
            if activation is not None:
                self.layers.append(activation())

    def forward(self, X):
        out = X
        for layer in self.layers:
            out = layer.forward(out)
        return out

    def backward(self, dout):
        for layer in reversed(self.layers):
            dout = layer.backward(dout)

    def update_weights(self):
        for layer in self.layers:
            if isinstance(layer, FC):
                layer.update()

    def compute_loss(self, X, y):
        y_pred = self.forward(X)
        return self.loss_fn.forward(y_pred, y)

    def train(self, X_train, y_train, epochs=10, batch_size=32):
        # Convert to NumPy arrays if they're not already
        X_train = X_train.to_numpy() if isinstance(X_train, pd.DataFrame) else X_train
        y_train = y_train.to_numpy() if isinstance(y_train, pd.Series) else y_train
        
        n_samples = X_train.shape[0]
        
        for epoch in range(epochs):
            # Shuffle the dataset
            idx = np.random.permutation(n_samples)
            X_train = X_train[idx]
            y_train = y_train[idx]

            # Train in batches
            for i in range(0, n_samples, batch_size):
                X_batch = X_train[i:i + batch_size]
                y_batch = y_train[i:i + batch_size]

                # Forward pass
                out = self.forward(X_batch)
                
                # Compute loss
                loss = self.loss_fn.forward(out, y_batch)
                
                # Backward pass
                dout = self.loss_fn.backward()
                self.backward(dout)
                
                # Update weights
                self.update_weights()

            print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss:.4f}")

    def predict(self, X):
        out = self.forward(X)
        return np.argmax(out, axis=1)


# Verification
# Problem 9: Learning and estimation
Create several networks with different numbers of layers and activation functions, train and predict the MNIST data, and calculate the accuracy.

In [15]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score
import numpy as np

# Load MNIST
mnist = fetch_openml('mnist_784', version=1)
X = mnist.data / 255.0  # Normalize
y = mnist.target.astype(int).to_numpy().reshape(-1, 1)

# One-hot encode labels
encoder = OneHotEncoder(sparse_output=False)
y_encoded = encoder.fit_transform(y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

/usr/local/lib/python3.11/dist-packages/sklearn/datasets/_openml.py:968: FutureWarning: The default value of `parser` will change from `'liac-arff'` to `'auto'` in 1.4. You can set `parser='auto'` to silence this warning. Therefore, an `ImportError` will be raised from 1.4 if the dataset is dense and pandas is not installed. Note that the pandas parser may return different data types. See the Notes Section in fetch_openml's API doc for details.
  warn(


In [16]:
# Define neural network configuration
config = [
    (784, 128, ReLU),
    (128, 64, ReLU),
    (64, 10, None)  # No activation after final layer
]

In [17]:
# Initialize components
initializer = HeInitializer()
optimizer = AdaGrad(lr=0.01)
loss_fn = SoftmaxWithLoss()

# Create model
model = ScratchDeepNeuralNetworkClassifier(config, initializer, optimizer, loss_fn)

# Train
model.train(X_train, y_train, epochs=10, batch_size=64)

Epoch 1/10, Loss: 0.1507
Epoch 2/10, Loss: 0.0716
Epoch 3/10, Loss: 0.0337
Epoch 4/10, Loss: 0.0877
Epoch 5/10, Loss: 0.0697
Epoch 6/10, Loss: 0.0164
Epoch 7/10, Loss: 0.0568
Epoch 8/10, Loss: 0.0569
Epoch 9/10, Loss: 0.0300
Epoch 10/10, Loss: 0.0166


In [18]:
# Predict
y_pred = model.predict(X_test)
y_true = np.argmax(y_test, axis=1)

# Accuracy
accuracy = accuracy_score(y_true, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.9709


The accuracy on the MNIST test set is 97.2% showing that the model ran successfully.

# Summary
1. Data Preprocessing: load data, normalize pixels to range from 0 to 1, on-hot encoded the target labels to be suitable for multiclass classification, split dataset.
2. Neural Network Configuration: for a three-layer neural network (784 - 128, 128 - 64 & 64 - 10)
3. Model Initialization: Used HeInitializer to initialize weights, AdaGrad as the optimizer, ppplied SoftmaxWithLoss for the loss function to handle multi-class classification.
4. Training: Trained the model on the training data for 10 epochs with a batch size of 64 then printed the loss after each epoch for progress tracking.
5. Evaluation: Used trained model to predict the test set labels and calculated accuracy of the predictions.